# ⏱️ Day 03a: Latency & Throughput — Numbers Every Programmer Should Know

---

## 🎯 What You'll Master Today
- Latency numbers at every level (L1 cache to cross-continental)
- Throughput calculations
- P99 latency & SLAs
- Latency vs throughput trade-offs

**Goal:** Quote latency numbers from memory in interviews.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  LATENCY NUMBERS EVERY PROGRAMMER SHOULD KNOW (2024)           ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  L1 cache reference .................... 1 ns                   ║
║  L2 cache reference .................... 4 ns                   ║
║  Main memory (RAM) .................... 100 ns                  ║
║  SSD random read ..................... 16 μs  (16,000 ns)      ║
║  Read 1 MB from memory .............. 250 μs                   ║
║  Read 1 MB from SSD ................. 1 ms                     ║
║  HDD seek ........................... 4 ms                     ║
║  Read 1 MB from HDD ................. 2 ms                     ║
║  Read 1 MB from network ............. 10 ms                    ║
║  Roundtrip same datacenter ........... 0.5 ms                  ║
║  Roundtrip US East → West ........... 40 ms                   ║
║  Roundtrip US → Europe .............. 80 ms                    ║
║  Roundtrip US → India ............... 150 ms                   ║
║                                                                   ║
║  💡 Memory = 100 ns, SSD = 16 μs (160× slower)                ║
║     SSD = 16 μs, Network = 500 μs (30× slower)                ║
║                                                                   ║
║  Rule of thumb: RAM → μs, SSD → μs-ms, Network → ms          ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. Latency Scale Visualization
# ============================================================

# If L1 cache = 1 second, what would everything else be?
scale_factor = 1e9  # 1 ns → 1 second

latencies = [
    ("L1 cache", 1, "ns"),
    ("L2 cache", 4, "ns"),
    ("RAM", 100, "ns"),
    ("SSD read", 16_000, "ns"),
    ("1MB from memory", 250_000, "ns"),
    ("1MB from SSD", 1_000_000, "ns"),
    ("HDD seek", 4_000_000, "ns"),
    ("Same datacenter RT", 500_000, "ns"),
    ("1MB from network", 10_000_000, "ns"),
    ("US East→West RT", 40_000_000, "ns"),
    ("US→Europe RT", 80_000_000, "ns"),
]

print(f"{'Operation':<25} {'Actual':>12} {'If L1=1 second':>20}")
print(f"{'─'*25} {'─'*12} {'─'*20}")
for name, ns, unit in latencies:
    scaled = ns  # ns * (1s / 1ns) = seconds
    if scaled < 60:
        human = f"{scaled} seconds"
    elif scaled < 3600:
        human = f"{scaled/60:.0f} minutes"
    elif scaled < 86400:
        human = f"{scaled/3600:.1f} hours"
    elif scaled < 86400 * 365:
        human = f"{scaled/86400:.0f} days"
    else:
        human = f"{scaled/(86400*365):.1f} years"
    
    if ns < 1000:
        actual = f"{ns} ns"
    elif ns < 1_000_000:
        actual = f"{ns/1000:.0f} μs"
    else:
        actual = f"{ns/1_000_000:.0f} ms"
    
    print(f"{name:<25} {actual:>12} {human:>20}")

print("\n💡 Helps build intuition: RAM is fast, disk is slow, network is VERY slow")

In [ ]:
# ============================================================
# 2. Throughput — Requests Per Second (QPS)
# ============================================================

print("""
THROUGHPUT REFERENCE NUMBERS
═══════════════════════════

Component Throughput:
  ┌────────────────────────┬──────────────────────────────┐
  │ Component              │ Typical QPS / Throughput     │
  ├────────────────────────┼──────────────────────────────┤
  │ Single web server      │ 1,000 - 10,000 QPS          │
  │ MySQL (reads)          │ 5,000 - 10,000 QPS          │
  │ MySQL (writes)         │ 1,000 - 5,000 QPS           │
  │ Redis (reads)          │ 100,000+ QPS                 │
  │ Memcached              │ 100,000+ QPS                 │
  │ Kafka (single broker)  │ 100,000 messages/sec         │
  │ Nginx (static)         │ 100,000+ QPS                 │
  └────────────────────────┴──────────────────────────────┘

  💡 Redis/Cache: 100K QPS. DB: 5-10K QPS.
     This is why caching gives 10-20× improvement!
""")

# Calculate: daily active users to QPS
def dau_to_qps(dau, requests_per_user=10):
    """Convert daily active users to queries per second."""
    daily_requests = dau * requests_per_user
    qps = daily_requests / 86400  # seconds in a day
    peak_qps = qps * 3  # peak is typically 2-3x average
    return qps, peak_qps

for dau in [100_000, 1_000_000, 10_000_000, 100_000_000]:
    avg, peak = dau_to_qps(dau)
    print(f"  {dau:>12,} DAU → {avg:>8,.0f} avg QPS, {peak:>8,.0f} peak QPS")

In [ ]:
# ============================================================
# 3. P99, P95, P50 — Percentile Latency
# ============================================================

print("""
╔═══════════════════════════════════════════════════════════════╗
║  PERCENTILE LATENCY                                          ║
╠═══════════════════════════════════════════════════════════════╣
║                                                               ║
║  P50 (median): 50% of requests faster than this             ║
║  P95:          95% faster (tail latency starts here)        ║
║  P99:          99% faster (worst-case for most users)       ║
║  P99.9:        99.9% (only 1 in 1000 is slower)            ║
║                                                               ║
║  Example API:                                                ║
║    P50 = 50ms (typical user experience)                     ║
║    P95 = 200ms (still acceptable)                           ║
║    P99 = 800ms (1% of users, often VIPs w/ more data)      ║
║                                                               ║
║  Why P99 matters:                                            ║
║    • 1% of requests × 1M users = 10,000 bad experiences    ║
║    • Your biggest customers often have the most data        ║
║    • One slow service in a chain slows everything           ║
║                                                               ║
║  SLA examples:                                               ║
║    99.9% = 8.7 hours downtime/year                          ║
║    99.99% = 52 minutes/year                                  ║
║    99.999% = 5 minutes/year ("five nines")                  ║
║                                                               ║
╚═══════════════════════════════════════════════════════════════╝
""")

# SLA calculation
for nines in [99, 99.9, 99.99, 99.999]:
    downtime_mins = (1 - nines/100) * 365 * 24 * 60
    if downtime_mins > 60:
        print(f"  {nines}% uptime → {downtime_mins/60:.1f} hours/year downtime")
    else:
        print(f"  {nines}% uptime → {downtime_mins:.1f} minutes/year downtime")

In [ ]:
# ============================================================
# 4. Latency vs Throughput Trade-offs
# ============================================================

print("""
LATENCY vs THROUGHPUT — They're Different!
═══════════════════════════════════════════

Latency  = time for ONE request (ms)
Throughput = number of requests per second (QPS)

Analogy: Highway
  Latency   = how long to drive from A to B
  Throughput = how many cars reach B per hour

Strategies for each:

  ┌─────────────────────┬─────────────────────────────────┐
  │ Reduce LATENCY      │ Increase THROUGHPUT             │
  ├─────────────────────┼─────────────────────────────────┤
  │ Cache hot data       │ Horizontal scaling              │
  │ CDN (closer to user)│ Async processing (queues)       │
  │ Pre-compute results │ Connection pooling              │
  │ Reduce hops         │ Batch operations                │
  │ Better algorithms   │ Database read replicas          │
  │ Compress responses  │ Load balancing                  │
  │ Keep-alive conns    │ Parallel processing             │
  └─────────────────────┴─────────────────────────────────┘

  💡 In interviews: "For this feature, are we optimizing for
     latency (user experience) or throughput (system capacity)?"
""")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | What latency numbers should you know? | RAM=100ns, SSD=16μs, Same DC=0.5ms, Cross-country=40ms. Memory 1000× faster than disk |
| 2 | Latency vs throughput? | Latency=time for one request. Throughput=requests per second. Different optimization strategies |
| 3 | Why does P99 matter? | 1% × 1M users = 10K bad experiences. Big customers often hit P99. One slow service slows the chain |
| 4 | How many QPS can a single server handle? | Web server: 1-10K. MySQL: 5-10K reads. Redis: 100K+. Depends on complexity |
| 5 | How to reduce latency? | Cache, CDN, pre-compute, fewer hops, compress, keep-alive connections, better algorithms |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • RAM=100ns, SSD=16μs, Network DC=0.5ms               │
## │  • Cache (Redis) = 100K QPS, DB = 5-10K QPS             │
## │  • P99 matters more than average latency                │
## │  • 99.99% = 52 min downtime/year                        │
## │  • Latency ≠ throughput — optimize for the right one    │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Back-of-Envelope Estimation** — the SD interview superpower